In [2]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("TMDB_API_KEY")

if not api_key:
    raise RuntimeError("TMDB_API_KEY is missing from .env")

response = requests.get(
    "https://api.themoviedb.org/3/search/movie",
    params={
        "api_key": api_key,
        "query": "Mulholland Drive",
        "language": "en-US",
    },
    timeout=15,
)

response.raise_for_status()

for movie in response.json().get("results", [])[:1]:
    print(
        {
            "id": movie["id"],
            "title": movie["title"],
            "release_date": movie.get("release_date"),
            "overview": movie.get("overview"),
            "vote_average": movie.get("vote_average"),
        }
    )

{'id': 1018, 'title': 'Mulholland Drive', 'release_date': '2001-06-06', 'overview': "Blonde Betty Elms has only just arrived in Hollywood to become a movie star when she meets an enigmatic brunette with amnesia. Meanwhile, as the two set off to solve the second woman's identity, filmmaker Adam Kesher runs into ominous trouble while casting his latest project.", 'vote_average': 7.799}


In [6]:
result = response.json()
len(result)
result

{'page': 1,
 'results': [{'adult': False,
   'backdrop_path': '/g9ClaS5EqU0Hk8WIbMNhld0Bk6f.jpg',
   'genre_ids': [53, 18, 9648],
   'id': 1018,
   'title': 'Mulholland Drive',
   'original_language': 'en',
   'original_title': 'Mulholland Drive',
   'overview': "Blonde Betty Elms has only just arrived in Hollywood to become a movie star when she meets an enigmatic brunette with amnesia. Meanwhile, as the two set off to solve the second woman's identity, filmmaker Adam Kesher runs into ominous trouble while casting his latest project.",
   'popularity': 16.8765,
   'poster_path': '/x7A59t6ySylr1L7aubOQEA480vM.jpg',
   'release_date': '2001-06-06',
   'softcore': False,
   'video': False,
   'vote_average': 7.799,
   'vote_count': 7225},
  {'adult': False,
   'backdrop_path': None,
   'genre_ids': [99],
   'id': 669330,
   'title': 'Return to Mulholland Drive',
   'original_language': 'fr',
   'original_title': 'Retour à Mulholland Drive',
   'overview': "French documentary about David 

In [7]:
movie_id = response.json()["results"][0]["id"]

details_response = requests.get(
    f"https://api.themoviedb.org/3/movie/{movie_id}",
    params={
        "api_key": api_key,
        "language": "en-US",
        "append_to_response": "credits,similar",
    },
    timeout=15,
)

details_response.raise_for_status()

movie = details_response.json()

director = next(
    (
        person["name"]
        for person in movie["credits"]["crew"]
        if person["job"] == "Director"
    ),
    None,
)

print(
    {
        "id": movie["id"],
        "title": movie["title"],
        "release_date": movie.get("release_date"),
        "runtime_minutes": movie.get("runtime"),
        "genres": [genre["name"] for genre in movie.get("genres", [])],
        "director": director,
        "top_cast": [
            person["name"]
            for person in movie["credits"].get("cast", [])[:5]
        ],
        "overview": movie.get("overview"),
        "tmdb_vote_average": movie.get("vote_average"),
        "tmdb_vote_count": movie.get("vote_count"),
        "similar_films": [
            item["title"]
            for item in movie.get("similar", {}).get("results", [])[:5]
        ],
    }
)

{'id': 1018, 'title': 'Mulholland Drive', 'release_date': '2001-06-06', 'runtime_minutes': 147, 'genres': ['Thriller', 'Drama', 'Mystery'], 'director': 'David Lynch', 'top_cast': ['Naomi Watts', 'Laura Harring', 'Justin Theroux', 'Ann Miller', 'Mark Pellegrino'], 'overview': "Blonde Betty Elms has only just arrived in Hollywood to become a movie star when she meets an enigmatic brunette with amnesia. Meanwhile, as the two set off to solve the second woman's identity, filmmaker Adam Kesher runs into ominous trouble while casting his latest project.", 'tmdb_vote_average': 7.799, 'tmdb_vote_count': 7222, 'similar_films': ['After Darkness', 'The Anchor', 'I, John Wayne', 'Malice', 'Eastern Promises']}
